# 01 Baseline Evaluation and Dataset Preparation

> Nemotron SFT-RL 课程大作业 · 第 1 个 Notebook
>
> 本 Notebook 是为 `datawhalechina/llm-algo-leetcode` 后训练优化路线准备的参考实现。它先固定真实数据身份、分类、划分和泄漏审计，再提供一个默认关闭的 Qwen3/vLLM Baseline 入口，以及可选的数据合成与筛查 Pilot。

**当前证据状态**：数据读取、分类、清洗、IID 划分、审计和合成 smoke test 可在无模型下载条件下验证；Qwen3 模型权重和 vLLM GPU 推理只有在目标 RTX 5090 与官方 vLLM Profile 准备好后才运行。Notebook 不在执行过程中安装依赖。当前权重位于 AutoDL 数据盘；下载时应先执行 `/etc/network_turbo` 学术加速入口。


## 学习目标

完成本节后，你应当能够：

1. 从官方 `train.csv` 的实际样本归纳可审计的题型分类，而不是依赖旧统计。
2. 将 Equation Numeric 与 Equation Symbolic 分开，并保留所有被排除样本的审计记录。
3. 在固定随机种子下生成 Train / Validation / Holdout Test，并检查精确重复、规范化重复和跨集合污染。
4. 理解原始模型 Baseline 的边界：原始 Prompt、官方 Chat Template、统一解码和确定性答案验证。
5. 在已确认的先验范围内构造小型合成样本；Equation Numeric 使用明确的 24 条规则，Guess 只有在全局约束下答案唯一时才进入合格数据。


## 1. 实验约束与运行配置

首版只研究三个真实题型：Gravity、Equation Numeric、Bit Manipulation。Gravity 是能力保持控制组，Equation Numeric 和 Bit Manipulation 是后续 RL 的主要任务。模型固定为 `Qwen/Qwen3-4B-Thinking-2507`，训练和 vLLM Rollout 使用 Datawhale 官方隔离 Profile。

`RUN_BASELINE=False` 是安全默认值：它不会下载模型或占用 GPU。准备好官方 vLLM Profile 后，将其改为 `True`，并先运行 `smoke`。AutoDL 目标机使用 `/root/autodl-tmp` 数据盘；模型可通过 `source /etc/network_turbo` 后下载到该盘，再以本地路径加载。


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import re
import time
from collections import Counter
from decimal import Decimal, InvalidOperation
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
RAW_CANDIDATES = [
    PROJECT_ROOT / "train.csv",
    PROJECT_ROOT.parent / "train.csv",
]
RAW_PATH = next((p for p in RAW_CANDIDATES if p.exists()), None)
if RAW_PATH is None:
    raise FileNotFoundError(f"未找到 train.csv，检查路径：{RAW_CANDIDATES}")

OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
SPLIT_DIR = OUTPUT_DIR / "splits"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
RUN_MODE = "smoke"                 # smoke / full
MAX_SAMPLES_PER_TASK = 20
SEED = 42
RUN_BASELINE = False               # 默认不下载模型
RUN_SYNTHETIC = False              # True 时才生成每类 100 条 Pilot
SYNTHETIC_SAMPLES_PER_TASK = 100
SOURCE_DATASET = "NVIDIA Nemotron Model Reasoning Challenge / train.csv"

TARGET_TASKS = {"gravity", "equation_numeric", "bit_manipulation"}
print({"raw_path": str(RAW_PATH), "output_dir": str(OUTPUT_DIR), "model_id": MODEL_ID, "run_baseline": RUN_BASELINE})


## 2. 读取并核验官方数据

这里使用字符串读取，避免 ID、二进制答案或带前导零的数值被自动转换。`source_row` 是原始 CSV 中的行号（表头为第 1 行），后续所有产物都保留它和原始 `id`。


In [ ]:
raw_df = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)
required_columns = ["id", "prompt", "answer"]
missing_columns = [c for c in required_columns if c not in raw_df.columns]
if missing_columns:
    raise ValueError(f"缺少字段：{missing_columns}")
raw_df = raw_df[required_columns].copy()
raw_df.insert(0, "source_row", np.arange(2, len(raw_df) + 2, dtype=int))
raw_df["source_dataset"] = SOURCE_DATASET

empty_counts = {c: int((raw_df[c].str.len() == 0).sum()) for c in required_columns}
raw_profile = {
    "sha256": hashlib.sha256(RAW_PATH.read_bytes()).hexdigest(),
    "rows": int(len(raw_df)),
    "columns": list(raw_df.columns),
    "dtypes": {c: str(raw_df[c].dtype) for c in raw_df.columns},
    "empty_counts": empty_counts,
    "null_counts": {c: int(raw_df[c].isna().sum()) for c in raw_df.columns},
    "duplicate_id_rows": int(raw_df["id"].duplicated(keep=False).sum()),
    "duplicate_id_repeats": int(raw_df["id"].duplicated().sum()),
    "duplicate_prompt_rows": int(raw_df["prompt"].duplicated(keep=False).sum()),
    "duplicate_prompt_repeats": int(raw_df["prompt"].duplicated().sum()),
}
print(json.dumps(raw_profile, ensure_ascii=False, indent=2))
print("\n首行字段：", raw_df.iloc[0][required_columns].to_dict())


## 3. 题型分类：使用精确模板，再细分 Equation

官方数据的六个首句模板可以被精确识别：Gravity、Bit Manipulation、Equation、Unit Conversion、Encryption 和 Numeral System。Equation 模板内部必须再分：含数字的样本是 Equation Numeric，完全由符号构成的样本是 Equation Symbolic。

这里不使用宽泛的 `"equation" in prompt` 规则。遇到没有匹配模板的样本会进入 `unknown`，不会静默丢弃。Numeric 样本还会根据 Query 运算符是否在示例中出现，标记 `deduce` 或 `guess`。


In [ ]:
TEMPLATE_PREFIXES = [
    ("gravity", "In Alice's Wonderland, the gravitational constant has been secretly changed.", "exact_gravity_prefix"),
    ("bit_manipulation", "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers.", "exact_bit_prefix"),
    ("equation", "In Alice's Wonderland, a secret set of transformation rules is applied to equations.", "exact_equation_prefix"),
    ("unit_conversion", "In Alice's Wonderland, a secret unit conversion is applied to measurements.", "exact_unit_prefix"),
    ("encryption", "In Alice's Wonderland, secret encryption rules are used on text.", "exact_encryption_prefix"),
    ("numeral_system", "In Alice's Wonderland, numbers are secretly converted into a different numeral system.", "exact_numeral_prefix"),
]


def normalize_prompt(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"0b[01]+", "<bin>", text)
    text = re.sub(r"[-+]?\d+(?:\.\d+)?", "<num>", text)
    text = re.sub(r"\s+", " ", text)
    return text


def prompt_hash(text: str) -> str:
    return hashlib.sha256(str(text).encode("utf-8")).hexdigest()


def query_operator_and_example_operators(prompt: str) -> tuple[str | None, set[str]]:
    example_ops = set(re.findall(r"(?m)^\s*\d+\s*(.)\s*\d+\s*=", prompt))
    query = re.search(r"Now, determine the result for:\s*([0-9]+)\s*(.)\s*([0-9]+)", prompt)
    if query is None:
        return None, example_ops
    return query.group(2), example_ops


def classify_prompt(prompt: str) -> dict[str, object]:
    prompt = str(prompt)
    for base_label, prefix, rule in TEMPLATE_PREFIXES:
        if prompt.startswith(prefix):
            if base_label == "equation":
                if re.search(r"\d", prompt):
                    query_op, example_ops = query_operator_and_example_operators(prompt)
                    subtype = "deduce" if query_op in example_ops else "guess"
                    if query_op is None:
                        subtype = "undetermined"
                    return {
                        "task_type": "equation_numeric",
                        "equation_subtype": subtype,
                        "classification_confidence": "high",
                        "classification_rule": "exact_equation_prefix_plus_digit_presence",
                        "query_operator": query_op or "",
                        "example_operators": "".join(sorted(example_ops)),
                    }
                return {
                    "task_type": "equation_symbolic",
                    "equation_subtype": "symbolic",
                    "classification_confidence": "high",
                    "classification_rule": "exact_equation_prefix_plus_no_digits",
                    "query_operator": "",
                    "example_operators": "",
                }
            return {
                "task_type": base_label,
                "equation_subtype": "",
                "classification_confidence": "high",
                "classification_rule": rule,
                "query_operator": "",
                "example_operators": "",
            }
    return {
        "task_type": "unknown",
        "equation_subtype": "",
        "classification_confidence": "low",
        "classification_rule": "no_registered_template_prefix",
        "query_operator": "",
        "example_operators": "",
    }


classification_records = []
for row in raw_df.itertuples(index=False):
    result = classify_prompt(row.prompt)
    normalized = normalize_prompt(row.prompt)
    classification_records.append({
        "source_row": int(row.source_row),
        "source_dataset": row.source_dataset,
        "id": row.id,
        "original_prompt": row.prompt,
        "original_answer": row.answer,
        "prompt_hash": prompt_hash(row.prompt),
        "normalized_prompt_hash": prompt_hash(normalized),
        "has_digit": bool(re.search(r"\d", row.prompt)),
        "prompt_characters": int(len(row.prompt)),
        "answer_characters": int(len(row.answer)),
        **result,
        "kept_for_course": result["task_type"] in TARGET_TASKS,
        "exclusion_reason": "" if result["task_type"] in TARGET_TASKS else f"excluded_{result['task_type']}",
    })

audit_df = pd.DataFrame(classification_records)
print(audit_df["task_type"].value_counts(dropna=False).to_string())
print("\nEquation Numeric subtype:")
print(audit_df.loc[audit_df.task_type == "equation_numeric", "equation_subtype"].value_counts(dropna=False).to_string())


In [ ]:
print("各分类抽样（只显示 Prompt 首句和答案，不修改原始数据）：")
for task_type, group in raw_df.assign(task_type=audit_df["task_type"]).groupby("task_type", sort=True):
    print(f"\n### {task_type}: n={len(group)}")
    sample = group.sample(n=min(2, len(group)), random_state=SEED)
    for row in sample.itertuples(index=False):
        print({"id": row.id, "prompt_head": row.prompt.splitlines()[0], "answer": row.answer})

unknown_rows = audit_df[audit_df.task_type == "unknown"]
if len(unknown_rows):
    print("\nUNKNOWN 样本必须人工复核：")
    print(unknown_rows[["source_row", "id", "classification_rule"]].head(20).to_string(index=False))
else:
    print("\n没有未知模板；仍保留 unknown 分类路径以防数据版本改变。")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
counts = audit_df["task_type"].value_counts().sort_values(ascending=True)
counts.plot.barh(ax=ax, color="#4C78A8")
ax.set_title("Raw task distribution (exact template classification)")
ax.set_xlabel("Number of samples")
ax.set_ylabel("Task type")
for i, value in enumerate(counts):
    ax.text(value + max(counts) * 0.01, i, str(value), va="center")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "raw_task_distribution.png", dpi=150)
plt.show()


## 4. 清洗主线：只保留三个正式题型

清洗只保留 `gravity`、`equation_numeric` 和 `bit_manipulation`。`equation_symbolic`、Unit Conversion、Encryption、Numeral System 和 Unknown 都保留在 `classification_audit.csv` 中，并带有排除原因。

`real_cleaned.csv` 仍保留原始 ID、原始 Prompt、原始 Answer 和来源行号，后续训练数据可以从它追溯回官方文件。


In [ ]:
raw_with_classification = raw_df.join(audit_df.set_index("source_row"), on="source_row", rsuffix="_audit")
cleaned_df = raw_with_classification[raw_with_classification["task_type"].isin(sorted(TARGET_TASKS))].copy()
cleaned_columns = [
    "source_row", "source_dataset", "id", "prompt", "answer", "task_type", "equation_subtype",
    "query_operator", "example_operators", "prompt_hash", "normalized_prompt_hash",
    "classification_confidence", "classification_rule",
]
audit_path = OUTPUT_DIR / "classification_audit.csv"
cleaned_path = OUTPUT_DIR / "real_cleaned.csv"
audit_df.to_csv(audit_path, index=False)
cleaned_df[cleaned_columns].to_csv(cleaned_path, index=False)

cleaning_report = {
    "raw_rows": int(len(raw_df)),
    "kept_rows": int(len(cleaned_df)),
    "removed_rows": int(len(raw_df) - len(cleaned_df)),
    "kept_counts": {k: int(v) for k, v in cleaned_df.task_type.value_counts().to_dict().items()},
    "removed_counts": {k: int(v) for k, v in audit_df.loc[~audit_df.kept_for_course, "task_type"].value_counts().to_dict().items()},
    "audit_path": str(audit_path),
    "cleaned_path": str(cleaned_path),
}
print(json.dumps(cleaning_report, ensure_ascii=False, indent=2))


## 5. 划分与 Leakage Audit

默认划分是固定种子的**分层 IID Holdout**：每个正式题型分别按 80% / 10% / 10% 划分。它能提供同分布泛化观察，但不能自动证明不存在同源题目泄漏。

本数据没有公开的规则实例 ID、模板组 ID 或生成种子。我们会审计完全重复 Prompt 和把数字/二进制字面量替换后的规范化重复；如果规范化重复跨集合，只报告风险，不把它伪装成 Structural OOD 证据。Structural OOD 需要后续补充可靠的规则组合或结构家族字段后再做。


In [ ]:
def make_stratified_split(frame: pd.DataFrame, seed: int = SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    assignments = {}
    for task_type, group in frame.groupby("task_type", sort=True):
        indices = group.index.to_numpy(copy=True)
        rng.shuffle(indices)
        n = len(indices)
        n_validation = max(1, int(round(n * 0.10)))
        n_test = max(1, int(round(n * 0.10)))
        for index in indices[: n - n_validation - n_test]:
            assignments[index] = "train"
        for index in indices[n - n_validation - n_test : n - n_test]:
            assignments[index] = "validation"
        for index in indices[n - n_test :]:
            assignments[index] = "holdout_test"
    result = frame.copy()
    result["split"] = pd.Series(assignments)
    if result["split"].isna().any():
        raise AssertionError("存在未分配 split 的样本")
    return result

split_df = make_stratified_split(cleaned_df, seed=SEED)
for split_name, split_group in split_df.groupby("split", sort=True):
    split_group[cleaned_columns + ["split"]].to_csv(SPLIT_DIR / f"{split_name}.csv", index=False)


def cross_split_duplicate_counts(frame: pd.DataFrame, key: str) -> dict[str, object]:
    owners = frame.groupby(key)["split"].agg(lambda values: sorted(set(values)))
    leaked = owners[owners.map(len) > 1]
    return {
        "key": key,
        "duplicate_keys": int((owners.map(len) > 1).sum()),
        "rows_in_cross_split_keys": int(frame[frame[key].isin(leaked.index)].shape[0]),
        "examples": [{"key": str(k), "splits": v} for k, v in leaked.head(5).items()],
    }

split_counts = (
    split_df.groupby(["split", "task_type"], sort=True).size().unstack(fill_value=0).astype(int)
)
leakage_audit = {
    "exact_id": cross_split_duplicate_counts(split_df, "id"),
    "exact_prompt": cross_split_duplicate_counts(split_df, "prompt_hash"),
    "normalized_prompt": cross_split_duplicate_counts(split_df, "normalized_prompt_hash"),
    "group_aware_split": {
        "status": "not_available",
        "reason": "原始 CSV 未提供可靠的规则实例、生成种子或结构家族字段；规范化 Prompt 只能用于审计，不能作为同源组的充分证明。",
    },
}
print(split_counts.to_string())
print(json.dumps(leakage_audit, ensure_ascii=False, indent=2))


In [ ]:
def prompt_token_proxy(text: str) -> int:
    # 这是不依赖模型 Tokenizer 的审计量，不应冒充 Qwen3 token 数。
    return len(re.findall(r"\w+|[^\w\s]", str(text), flags=re.UNICODE))

split_df["prompt_token_proxy"] = split_df["prompt"].map(prompt_token_proxy)
rule_distribution = {
    "task_type": {k: int(v) for k, v in split_df.task_type.value_counts().to_dict().items()},
    "equation_subtype": {k: int(v) for k, v in split_df.loc[split_df.task_type == "equation_numeric", "equation_subtype"].value_counts().to_dict().items()},
    "equation_query_operator": {k: int(v) for k, v in split_df.loc[split_df.task_type == "equation_numeric", "query_operator"].value_counts().to_dict().items()},
}

dataset_statistics = {
    "schema_version": "nemotron-course-dataset/v1",
    "source": {"path": str(RAW_PATH.relative_to(PROJECT_ROOT)), "dataset": SOURCE_DATASET, "sha256": raw_profile["sha256"]},
    "seed": SEED,
    "raw_profile": raw_profile,
    "classification_counts": {k: int(v) for k, v in audit_df.task_type.value_counts().to_dict().items()},
    "cleaned_counts": {k: int(v) for k, v in cleaned_df.task_type.value_counts().to_dict().items()},
    "split_counts": {
        split: {task: int(value) for task, value in row.items()}
        for split, row in split_counts.iterrows()
    },
    "rule_distribution": rule_distribution,
    "length_summary": {
        split: {
            "rows": int(len(group)),
            "prompt_char_mean": float(group.prompt.str.len().mean()),
            "prompt_token_proxy_mean": float(group.prompt_token_proxy.mean()),
            "answer_char_mean": float(group.answer.str.len().mean()),
        }
        for split, group in split_df.groupby("split", sort=True)
    },
    "leakage_audit": leakage_audit,
    "limitations": [
        "prompt_token_proxy 不是 Qwen3 Tokenizer 的真实 token 数；真实 token 统计需在官方训练环境加载 tokenizer 后补充。",
        "没有可靠的规则实例/结构家族键，因此当前只提供 IID Holdout，不宣称 Structural OOD 已完成。",
        "规范化 Prompt 重复是风险信号，不等同于同源题目证明。",
    ],
}
(OUTPUT_DIR / "dataset_statistics.json").write_text(json.dumps(dataset_statistics, ensure_ascii=False, indent=2), encoding="utf-8")
split_manifest = {
    "schema_version": "nemotron-course-split/v1",
    "seed": SEED,
    "ratios_requested": {"train": 0.8, "validation": 0.1, "holdout_test": 0.1},
    "method": "per-task stratified IID split",
    "structural_ood": "not_available_without_reliable_group_key",
    "files": {name: str((SPLIT_DIR / f"{name}.csv").relative_to(PROJECT_ROOT)) for name in ["train", "validation", "holdout_test"]},
    "leakage_audit": leakage_audit,
}
(OUTPUT_DIR / "split_manifest.json").write_text(json.dumps(split_manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({"statistics": str(OUTPUT_DIR / 'dataset_statistics.json'), "split_manifest": str(OUTPUT_DIR / 'split_manifest.json')}, ensure_ascii=False, indent=2))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_counts = split_df.groupby(["split", "task_type"]).size().unstack(fill_value=0)
plot_counts.plot.bar(ax=ax, color=["#4C78A8", "#F58518", "#54A24B"])
ax.set_title("Stratified IID split by task type")
ax.set_xlabel("Split")
ax.set_ylabel("Number of samples")
ax.legend(title="Task type")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "split_task_distribution.png", dpi=150)
plt.show()


## 6. Baseline：原始 Prompt、统一生成和确定性验证

Baseline 必须回答“未经 Prompt Engineering、SFT 或 RL 的 Qwen3 基座模型表现如何”。因此：

- 输入只使用原始比赛 Prompt；不注入课程先验规则库。
- 生成使用 Qwen3 官方 Chat Template，并在模型支持时显式开启 Thinking。
- 评测输出保留原文、提取答案、提取状态、生成 token 数和耗时。
- 不把没有运行的分数写成结果。

答案提取器只负责把模型回答映射到候选答案；Verifier 再按题型比较。对无法提取或格式不合法的输出，计入 Invalid Output，而不是当作错误答案静默吞掉。


In [ ]:
def normalize_answer_text(value: object) -> str:
    text = str(value).strip()
    text = re.sub(r"^```(?:text|json)?\s*|\s*```$", "", text, flags=re.I)
    return text.strip()


def extract_final_answer(text: str, task_type: str) -> tuple[str | None, str]:
    text = str(text)
    boxed = re.findall(r"\\boxed\{([^{}]+)\}", text)
    if boxed:
        return normalize_answer_text(boxed[-1]), "boxed"
    tagged = re.findall(r"<(?:final|answer)>\s*(.*?)\s*</(?:final|answer)>", text, flags=re.I | re.S)
    if tagged:
        return normalize_answer_text(tagged[-1]), "tagged"
    if task_type == "bit_manipulation":
        matches = re.findall(r"(?<![01])[01]{8}(?![01])", text)
        if matches:
            return matches[-1], "binary_fallback"
    if task_type in {"gravity", "equation_numeric"}:
        matches = re.findall(r"(?<![A-Za-z])[-+]?\d+(?:\.\d+)?", text)
        if matches:
            return matches[-1], "numeric_fallback"
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    if lines:
        return normalize_answer_text(lines[-1]), "last_nonempty_line"
    return None, "empty"


def verify_answer(task_type: str, predicted: object, expected: object) -> tuple[bool, str]:
    if predicted is None:
        return False, "not_extracted"
    predicted_text = normalize_answer_text(predicted)
    expected_text = normalize_answer_text(expected)
    if task_type == "bit_manipulation":
        p = predicted_text.removeprefix("0b")
        e = expected_text.removeprefix("0b")
        if not re.fullmatch(r"[01]{8}", p):
            return False, "invalid_8bit_format"
        return p == e, "exact_8bit"
    if task_type in {"gravity", "equation_numeric"}:
        try:
            return Decimal(predicted_text) == Decimal(expected_text), "decimal_equivalence"
        except InvalidOperation:
            return False, "invalid_numeric_format"
    return predicted_text == expected_text, "exact_text"


def summarize_baseline_records(records: list[dict[str, object]]) -> dict[str, object]:
    frame = pd.DataFrame(records)
    if frame.empty:
        return {"records": 0, "status": "no_records"}
    summaries = {}
    for task_type, group in frame.groupby("task_type", sort=True):
        summaries[task_type] = {
            "records": int(len(group)),
            "accuracy": float(group["correct"].mean()),
            "answer_extraction_success_rate": float(group["extraction_success"].mean()),
            "invalid_output_rate": float((~group["extraction_success"]).mean()),
            "average_generation_tokens": float(group["generation_tokens"].mean()),
            "average_inference_time_seconds": float(group["inference_time_seconds"].mean()),
            "total_inference_time_seconds": float(group["inference_time_seconds"].sum()),
        }
    task_accuracy = frame.groupby("task_type")["correct"].mean()
    return {
        "records": int(len(frame)),
        "overall_accuracy_sample_weighted": float(frame["correct"].mean()),
        "macro_accuracy": float(task_accuracy.mean()),
        "by_task": summaries,
    }


In [ ]:
def render_qwen3_prompt(tokenizer, original_prompt: str) -> str:
    messages = [{"role": "user", "content": original_prompt}]
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=True
        )
    except TypeError:
        # 兼容旧版 Transformers：仍使用官方模板，但不传新参数。
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def run_vllm_baseline(
    evaluation_frame: pd.DataFrame,
    model_id: str = MODEL_ID,
    max_new_tokens: int = 512,
    output_path: Path = OUTPUT_DIR / "baseline_results.jsonl",
) -> dict[str, object]:
    """在官方独立 vLLM Profile 中运行 Baseline；函数本身不安装依赖。"""
    try:
        from transformers import AutoTokenizer
        from vllm import LLM, SamplingParams
    except ImportError as exc:
        return {"status": "blocked_missing_vllm_profile", "reason": repr(exc)}

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    prompts = [render_qwen3_prompt(tokenizer, p) for p in evaluation_frame["prompt"]]
    llm = LLM(model=model_id, tensor_parallel_size=1, gpu_memory_utilization=0.85)
    sampling_params = SamplingParams(temperature=0.0, top_p=1.0, max_tokens=max_new_tokens)
    started = time.perf_counter()
    outputs = llm.generate(prompts, sampling_params, use_tqdm=True)
    elapsed_total = time.perf_counter() - started

    records = []
    for row, request_output in zip(evaluation_frame.itertuples(index=False), outputs):
        generated = request_output.outputs[0]
        text = generated.text
        extracted, extraction_method = extract_final_answer(text, row.task_type)
        correct, verification_method = verify_answer(row.task_type, extracted, row.answer)
        token_ids = getattr(generated, "token_ids", ()) or ()
        records.append({
            "id": row.id,
            "source_row": int(row.source_row),
            "task_type": row.task_type,
            "prompt": row.prompt,
            "expected_answer": row.answer,
            "raw_output": text,
            "extracted_answer": extracted,
            "extraction_method": extraction_method,
            "verification_method": verification_method,
            "extraction_success": extracted is not None,
            "correct": bool(correct),
            "generation_tokens": int(len(token_ids)),
            "inference_time_seconds": float(elapsed_total / max(len(outputs), 1)),
        })
    output_path.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in records) + "\n", encoding="utf-8")
    summary = summarize_baseline_records(records)
    (OUTPUT_DIR / "baseline_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    return {"status": "completed", "summary": summary, "results_path": str(output_path)}


if RUN_BASELINE:
    baseline_frame = split_df[split_df["split"] == "holdout_test"].copy()
    if RUN_MODE == "smoke":
        baseline_frame = pd.concat([
            group.head(MAX_SAMPLES_PER_TASK) for _, group in baseline_frame.groupby("task_type", sort=True)
        ], ignore_index=True)
    baseline_status = run_vllm_baseline(baseline_frame)
else:
    baseline_status = {
        "status": "skipped_by_default",
        "reason": "RUN_BASELINE=False；未下载模型、未占用 GPU、未生成虚构分数。",
        "next_action": "在官方 INF-VLLM Profile 和 RTX 5090 可用时改为 True，并先运行 smoke。",
    }
print(json.dumps(baseline_status, ensure_ascii=False, indent=2))


## 7. 可选数据工程：Prior → Generator → Solver → Verifier → Screening

真实数据清洗与划分已经在前面的主线完成。这里使用固定种子从课程先验生成三类 Pilot，每类 100 条候选；合成过程只使用先验和 Train 侧规则空间，不读取 Validation / Holdout Test。

- **Gravity**：隐藏 `g`，Prompt 只给物理公式、示例和 Query；Verifier 在 9.50–19.50、0.01 步长的先验网格上独立检查答案唯一性。
- **Equation Numeric**：采用参考实现中的 24 条 `(parse, operation, format)` 规则。一个 Prompt 同时包含多个运算符；全局约束要求非 Join 规则处于同一 normal / flip family，并且每个非 Join semantic group 至多使用一次。Query 未出现在示例中时标为 Guess，只有 Query 输出仍唯一才接受。
- **Bit Manipulation**：使用课程规定的 8-bit DSL。Prompt 只呈现输入输出示例和 Query；Solver 枚举深度不超过 2 的表达式并按完整 256 输入语义去重，分别记录表达式唯一性和答案唯一性。

每条候选都保留 `hidden_rule`、参考 `answer`、`trace` 和 `verification_status`。这些训练元数据不进入 Prompt。Generator、Solver 和 Verifier 在代码中使用不同入口；拒绝候选保留拒绝原因。

### 可选 NeMo Data Designer 分支

根据 NVIDIA-NeMo/DataDesigner 官方 Python API 核对，接口为 `data_designer.config`、`data_designer.interface.DataDesigner`、`DataDesignerConfigBuilder.with_seed_dataset(LocalFileSeedSource(...))` 和 `DataDesigner.preview(builder, num_records=...)`。官方默认模型使用 `NVIDIA_API_KEY`，Notebook 不硬编码密钥，也不临时安装依赖。该分支只扩充 Prompt / 轨迹的语言表达，Ground Truth 仍由本 Notebook 的确定性 Solver 产生；Preview 结果必须再次经过 Verifier，默认关闭。若环境缺少依赖或密钥，运行记录会明确标记阻塞原因。


In [ ]:
# 合成分支的配置只使用课程先验和固定随机种子；不读取 Validation / Holdout Test。
from itertools import product
from functools import lru_cache
from datetime import datetime, timezone

SYNTHETIC_DIR = OUTPUT_DIR / "synthetic"
SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True)
PILOT_CANDIDATES_PER_TASK = 100
RUN_SYNTHETIC = True
RUN_DATA_DESIGNER_PREVIEW = False
GENERATOR_VERSION = "nemotron-synthetic-v2"

GRAVITY_PRIOR = {
    "formula": "distance = 0.5 * g * t^2",
    "units": {"t": "seconds", "distance": "meters", "g": "m/s^2"},
    "g_grid": {"min": 9.50, "max": 19.50, "step": 0.01},
    "answer_precision": "two_decimal_places",
}
BIT_PRIOR = {
    "width": 8,
    "operations": ["SHL", "SHR", "ROTL", "ROTR", "AND", "OR", "XOR", "NOT"],
    "expression_language": "x is the input byte; unary and binary trees have maximum depth 2",
    "mask": 255,
}

# Numeric Equation prior extracted from the reference solver's DSL and Numeric24 rule table.
# A rule is (operand parse, arithmetic operation, output format); it is not merely an
# operator -> answer lookup. Global assignments additionally enforce one family and one
# non-join semantic group per prompt, while join rules may coexist.
NUMERIC24_RULES = [
    {"name": "sub", "parse": "AB_CD", "operation": "sub_lr", "format": "raw", "family": "normal", "group": "sub_or_mod_like"},
    {"name": "absdiff", "parse": "AB_CD", "operation": "abs_sub", "format": "raw", "family": "normal", "group": "sub_or_mod_like"},
    {"name": "absdiff_prefix", "parse": "AB_CD", "operation": "abs_sub", "format": "sign_prefix_raw", "family": "normal", "group": "sub_or_mod_like"},
    {"name": "join", "parse": "AB_CD", "operation": "concat_lr", "format": "raw", "family": "join", "group": "join_like"},
    {"name": "join_swap", "parse": "AB_CD", "operation": "concat_rl", "format": "raw", "family": "join", "group": "join_like"},
    {"name": "add", "parse": "AB_CD", "operation": "add", "format": "raw", "family": "normal", "group": "add_like"},
    {"name": "add_p1", "parse": "AB_CD", "operation": "add_p1", "format": "raw", "family": "normal", "group": "add_like"},
    {"name": "add_m1", "parse": "AB_CD", "operation": "add_m1", "format": "raw", "family": "normal", "group": "add_like"},
    {"name": "mul", "parse": "AB_CD", "operation": "mul", "format": "raw", "family": "normal", "group": "mul_like"},
    {"name": "mul_p1", "parse": "AB_CD", "operation": "mul_p1", "format": "raw", "family": "normal", "group": "mul_like"},
    {"name": "mul_m1", "parse": "AB_CD", "operation": "mul_m1", "format": "raw", "family": "normal", "group": "mul_like"},
    {"name": "maxmodmin", "parse": "AB_CD", "operation": "max_mod_min", "format": "raw", "family": "normal", "group": "sub_or_mod_like"},
    {"name": "flip_add", "parse": "BA_DC", "operation": "add", "format": "rev", "family": "flip", "group": "add_like"},
    {"name": "flip_add_p1", "parse": "BA_DC", "operation": "add_p1", "format": "rev", "family": "flip", "group": "add_like"},
    {"name": "flip_add_m1", "parse": "BA_DC", "operation": "add_m1", "format": "rev", "family": "flip", "group": "add_like"},
    {"name": "flip_mul", "parse": "BA_DC", "operation": "mul", "format": "rev", "family": "flip", "group": "mul_like"},
    {"name": "flip_mul_p1", "parse": "BA_DC", "operation": "mul_p1", "format": "rev", "family": "flip", "group": "mul_like"},
    {"name": "flip_mul_m1", "parse": "BA_DC", "operation": "mul_m1", "format": "rev", "family": "flip", "group": "mul_like"},
    {"name": "flip_absdiff", "parse": "BA_DC", "operation": "abs_sub", "format": "rev", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_maxmodmin", "parse": "BA_DC", "operation": "max_mod_min", "format": "rev", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_sub_prefix", "parse": "BA_DC", "operation": "sub_lr", "format": "sign_prefix_rev", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_sub_suffix", "parse": "BA_DC", "operation": "sub_rl", "format": "rev_or_neg_suffix", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_absdiff_prefix", "parse": "BA_DC", "operation": "abs_sub", "format": "always_prefix_rev", "family": "flip", "group": "sub_or_mod_like"},
    {"name": "flip_absdiff_suffix", "parse": "BA_DC", "operation": "abs_sub", "format": "always_suffix_rev", "family": "flip", "group": "sub_or_mod_like"},
]
NUMERIC24_RULE_BY_NAME = {rule["name"]: rule for rule in NUMERIC24_RULES}
NUMERIC24_FORMULAS = {
    "sub": "AB - CD, raw", "absdiff": "|AB - CD|, raw", "absdiff_prefix": "|AB - CD| with operator prefix",
    "join": "AB concatenated with CD", "join_swap": "CD concatenated with AB",
    "add": "AB + CD", "add_p1": "AB + CD + 1", "add_m1": "AB + CD - 1",
    "mul": "AB * CD", "mul_p1": "AB * CD + 1", "mul_m1": "AB * CD - 1",
    "maxmodmin": "max(AB, CD) mod min(AB, CD)",
    "flip_add": "reverse(BA + DC)", "flip_add_p1": "reverse(BA + DC + 1)", "flip_add_m1": "reverse(BA + DC - 1)",
    "flip_mul": "reverse(BA * DC)", "flip_mul_p1": "reverse(BA * DC + 1)", "flip_mul_m1": "reverse(BA * DC - 1)",
    "flip_absdiff": "reverse(|BA - DC|)", "flip_maxmodmin": "reverse(max(BA, DC) mod min(BA, DC))",
    "flip_sub_prefix": "reverse(BA - DC), preserving a negative sign as an operator prefix",
    "flip_sub_suffix": "reverse(DC - BA), appending the operator for a negative result",
    "flip_absdiff_prefix": "operator-prefixed reverse(|BA - DC|)",
    "flip_absdiff_suffix": "reverse(|BA - DC|) followed by the operator",
}
EQUATION_NUMERIC_PRIOR = {
    "operand_pattern": "AB<operator>CD with two decimal digits on each side",
    "parses": {"AB_CD": "left=AB, right=CD", "BA_DC": "left=reverse(AB), right=reverse(CD)"},
    "operations": ["concatenate", "add", "add±1", "subtract", "reverse subtract", "absolute difference", "multiply", "multiply±1", "max mod min"],
    "output_formats": ["raw", "reverse digits", "operator-prefixed reverse", "two-digit padding", "operator suffix"],
    "rules": [{"name": name, "formula": NUMERIC24_FORMULAS[name]} for name in NUMERIC24_RULE_BY_NAME],
    "global_constraints": [
        "All non-join rules in one prompt use either normal or flipped operand family.",
        "At most one non-join rule from each semantic group: sub_or_mod_like, add_like, mul_like.",
        "Join and reverse-join are the join_like exception and can coexist with the other groups.",
        "A query operator absent from examples is Guess; it is admitted only when all valid assignments give one query output.",
    ],
}
print(json.dumps({"gravity_prior": GRAVITY_PRIOR, "bit_prior": BIT_PRIOR, "equation_numeric_rules": len(NUMERIC24_RULES), "pilot_candidates_per_task": PILOT_CANDIDATES_PER_TASK}, ensure_ascii=False, indent=2))


In [ ]:
# Gravity: generator, prompt-derived solver, and an independent verifier.
def _gravity_render(distance_t: float | Decimal, g_value: float | Decimal) -> str:
    t = Decimal(str(distance_t))
    g = Decimal(str(g_value))
    return str((Decimal("0.5") * g * t * t).quantize(Decimal("0.01")))


def _gravity_prompt_parts(prompt: str) -> tuple[list[tuple[str, str]], str | None]:
    examples = [(t, d) for t, d in re.findall(r"For t = ([0-9.]+)s, distance = ([0-9.]+) m", prompt)]
    query_match = re.search(r"falling distance for t = ([0-9.]+)s", prompt)
    return examples, query_match.group(1) if query_match else None


def solve_gravity_prompt(prompt: str) -> dict[str, object]:
    examples, query_t = _gravity_prompt_parts(prompt)
    if len(examples) < 2 or query_t is None:
        return {"status": "invalid_prompt", "answer": None, "inferred_g": None}
    estimates = [Decimal(distance) / (Decimal("0.5") * Decimal(t) * Decimal(t)) for t, distance in examples]
    inferred_g = (sum(estimates) / Decimal(len(estimates))).quantize(Decimal("0.01"))
    return {
        "status": "solved",
        "answer": _gravity_render(query_t, inferred_g),
        "inferred_g": str(inferred_g),
        "example_estimates": [str(value.quantize(Decimal("0.01"))) for value in estimates],
    }


def _gravity_grid_candidates(examples: list[tuple[str, str]]) -> list[Decimal]:
    candidates = []
    for cents in range(950, 1951):
        g_value = Decimal(cents) / Decimal(100)
        if all(_gravity_render(t, g_value) == str(distance) for t, distance in examples):
            candidates.append(g_value)
    return candidates


def gravity_identifiability(prompt: str) -> dict[str, object]:
    examples, query_t = _gravity_prompt_parts(prompt)
    candidates = _gravity_grid_candidates(examples)
    query_outputs = sorted({_gravity_render(query_t, g) for g in candidates}) if query_t else []
    return {
        "candidate_g_count": len(candidates),
        "candidate_g_values": [str(g) for g in candidates[:20]],
        "query_output_count": len(query_outputs),
        "query_outputs": query_outputs,
        "answer_unique": len(query_outputs) == 1,
    }


def generate_gravity_sample(rng: np.random.Generator, examples: int = 4) -> dict[str, object]:
    g_value = Decimal(int(rng.integers(950, 1951))) / Decimal(100)
    times = sorted(rng.choice(np.arange(60, 501), size=examples + 1, replace=False).tolist())
    times = [Decimal(int(value)) / Decimal(100) for value in times]
    example_pairs = [(str(t), _gravity_render(t, g_value)) for t in times[:-1]]
    query_t = times[-1]
    prompt = (
        "The course prior defines the physical relation distance = 0.5*g*t^2, with t in seconds and distance in meters.\n"
        + "\n".join(f"For t = {t}s, distance = {distance} m" for t, distance in example_pairs)
        + f"\nNow, calculate the falling distance for t = {query_t}s."
    )
    trace_steps = [
        {"t": t, "distance": distance, "g_estimate": str((Decimal(distance) / (Decimal("0.5") * Decimal(t) * Decimal(t))).quantize(Decimal("0.01")))}
        for t, distance in example_pairs
    ]
    return {
        "task_type": "gravity",
        "source": "deterministic_prior_generator",
        "generator_version": GENERATOR_VERSION,
        "hidden_rule": {"g": str(g_value), "formula": GRAVITY_PRIOR["formula"]},
        "prompt": prompt,
        "answer": _gravity_render(query_t, g_value),
        "trace": {"formula": GRAVITY_PRIOR["formula"], "examples": trace_steps, "query_t": str(query_t), "inferred_g": str(g_value)},
        "difficulty": "easy_control",
        "deduce_or_guess": "deduce",
    }


def verify_gravity_sample(sample: dict[str, object]) -> dict[str, object]:
    # This verifier recomputes the equation from prompt text and a finite prior grid;
    # it never reads hidden_rule to establish the answer.
    examples, query_t = _gravity_prompt_parts(str(sample.get("prompt", "")))
    candidates = _gravity_grid_candidates(examples)
    query_outputs = sorted({_gravity_render(query_t, g) for g in candidates}) if query_t else []
    answer = str(sample.get("answer", ""))
    answer_ok = answer in query_outputs
    trace = sample.get("trace", {})
    trace_ok = isinstance(trace, dict) and trace.get("formula") == GRAVITY_PRIOR["formula"] and trace.get("query_t") == query_t
    if trace_ok:
        for step in trace.get("examples", []):
            try:
                t, distance, g_estimate = step["t"], step["distance"], Decimal(str(step["g_estimate"]))
                trace_ok = trace_ok and _gravity_render(t, g_estimate) == str(distance)
            except (KeyError, InvalidOperation, TypeError):
                trace_ok = False
                break
    return {
        "answer_ok": answer_ok,
        "trace_ok": bool(trace_ok),
        "identifiability": {"candidate_g_count": len(candidates), "query_output_count": len(query_outputs), "answer_unique": len(query_outputs) == 1, "query_outputs": query_outputs},
        "difficulty_ok": str(sample.get("difficulty", "")).startswith("easy"),
        "solver_status": "independent_grid_verifier",
    }


In [ ]:
# Bit Manipulation: an 8-bit DSL with input/output-only prompts.
MASK8 = 0xFF
BIT_UNARY_OPS = ("NOT",)
BIT_SHIFT_OPS = ("SHL", "SHR", "ROTL", "ROTR")
BIT_BINARY_OPS = ("AND", "OR", "XOR")


def compute_bit_tree(node: tuple) -> int:
    kind = node[0]
    if kind == "lit":
        return int(node[1]) & MASK8
    if kind == "var":
        raise ValueError("uninstantiated_variable")
    if kind in BIT_SHIFT_OPS:
        value, shift = compute_bit_tree(node[1]), int(node[2]) % 8
        if kind == "SHL": return (value << shift) & MASK8
        if kind == "SHR": return (value >> shift) & MASK8
        if kind == "ROTL": return ((value << shift) | (value >> (8 - shift))) & MASK8 if shift else value
        return ((value >> shift) | (value << (8 - shift))) & MASK8 if shift else value
    if kind == "NOT":
        return (~compute_bit_tree(node[1])) & MASK8
    left, right = compute_bit_tree(node[1]), compute_bit_tree(node[2])
    return {"AND": left & right, "OR": left | right, "XOR": left ^ right}[kind]


def render_bit_tree(node: tuple) -> str:
    kind = node[0]
    if kind == "var": return "x"
    if kind == "lit": return f"0b{int(node[1]) & MASK8:08b}"
    if kind in BIT_SHIFT_OPS: return f"{kind}({render_bit_tree(node[1])}, {int(node[2])})"
    if kind == "NOT": return f"NOT({render_bit_tree(node[1])})"
    return f"{kind}({render_bit_tree(node[1])}, {render_bit_tree(node[2])})"


def instantiate_bit_tree(tree: tuple, value: int) -> tuple:
    """Instantiate every variable recursively, including both binary subtrees."""
    if tree[0] == "var":
        return ("lit", int(value) & MASK8)
    if tree[0] == "lit":
        return ("lit", int(tree[1]) & MASK8)
    if tree[0] in BIT_UNARY_OPS:
        return (tree[0], instantiate_bit_tree(tree[1], value))
    if tree[0] in BIT_SHIFT_OPS:
        return (tree[0], instantiate_bit_tree(tree[1], value), int(tree[2]) % 8)
    return (tree[0], instantiate_bit_tree(tree[1], value), instantiate_bit_tree(tree[2], value))


def random_bit_tree(rng: np.random.Generator, depth: int = 2) -> tuple:
    if depth <= 0:
        return ("var",)
    kind = str(rng.choice(BIT_SHIFT_OPS + BIT_BINARY_OPS + BIT_UNARY_OPS))
    left = random_bit_tree(rng, depth - 1)
    if kind in BIT_UNARY_OPS:
        return (kind, left)
    if kind in BIT_SHIFT_OPS:
        return (kind, left, int(rng.integers(0, 8)))
    return (kind, left, random_bit_tree(rng, depth - 1))


def _bit_tree_value(tree: tuple, value: int) -> int:
    return compute_bit_tree(instantiate_bit_tree(tree, value))


@lru_cache(maxsize=None)
def _bit_candidate_trees(max_depth: int = 2) -> tuple[tuple, ...]:
    by_depth = {0: (("var",),)}
    for depth in range(1, max_depth + 1):
        smaller = tuple(tree for d in range(depth) for tree in by_depth[d])
        candidates = []
        candidates.extend(("NOT", child) for child in smaller)
        candidates.extend((op, child, shift) for op in BIT_SHIFT_OPS for child in smaller for shift in range(8))
        candidates.extend((op, left, right) for op in BIT_BINARY_OPS for left in smaller for right in smaller)
        unique = {render_bit_tree(tree): tree for tree in candidates}
        by_depth[depth] = tuple(unique.values())
    all_trees = {render_bit_tree(tree): tree for depth in by_depth.values() for tree in depth}
    return tuple(all_trees.values())


def _bit_semantic_signature(tree: tuple) -> tuple[int, ...]:
    return tuple(_bit_tree_value(tree, value) for value in range(256))


def identify_bit_expression(example_pairs: list[tuple[int, int]], query_value: int, max_depth: int = 2) -> dict[str, object]:
    matching = []
    for tree in _bit_candidate_trees(max_depth):
        if all(_bit_tree_value(tree, value) == output for value, output in example_pairs):
            matching.append(tree)
    semantic_groups = {}
    for tree in matching:
        semantic_groups.setdefault(_bit_semantic_signature(tree), tree)
    query_outputs = sorted({_bit_tree_value(tree, query_value) for tree in semantic_groups.values()})
    return {
        "raw_candidate_count": len(matching),
        "semantic_candidate_count": len(semantic_groups),
        "query_output_count": len(query_outputs),
        "query_outputs": [f"{value:08b}" for value in query_outputs],
        "expression_unique": len(matching) == 1,
        "answer_unique": len(query_outputs) == 1,
    }


def _parse_bit_expression(text: str, position: int = 0) -> tuple[tuple, int]:
    while position < len(text) and text[position].isspace(): position += 1
    if text.startswith("x", position): return ("var",), position + 1
    literal = re.match(r"0b([01]{8})", text[position:])
    if literal: return ("lit", int(literal.group(1), 2)), position + literal.end()
    operation = re.match(r"(NOT|SHL|SHR|ROTL|ROTR|AND|OR|XOR)\(", text[position:])
    if operation is None: raise ValueError("invalid_bit_expression")
    op, pos = operation.group(1), position + operation.end()
    left, pos = _parse_bit_expression(text, pos)
    while pos < len(text) and text[pos].isspace(): pos += 1
    if op == "NOT":
        if pos >= len(text) or text[pos] != ")": raise ValueError("invalid_not")
        return (op, left), pos + 1
    if pos >= len(text) or text[pos] != ",": raise ValueError("missing_comma")
    pos += 1
    if op in BIT_SHIFT_OPS:
        match = re.match(r"\s*([0-7])\s*\)", text[pos:])
        if match is None: raise ValueError("invalid_shift")
        return (op, left, int(match.group(1))), pos + match.end()
    right, pos = _parse_bit_expression(text, pos)
    while pos < len(text) and text[pos].isspace(): pos += 1
    if pos >= len(text) or text[pos] != ")": raise ValueError("missing_close")
    return (op, left, right), pos + 1


def _evaluate_bit_expression(text: str, value: int) -> int:
    tree, position = _parse_bit_expression(text)
    if text[position:].strip(): raise ValueError("trailing_expression")
    return _bit_tree_value(tree, value)


def generate_bit_sample(rng: np.random.Generator, examples: int = 5) -> dict[str, object]:
    for _ in range(300):
        tree = random_bit_tree(rng, depth=2)
        inputs = rng.choice(np.arange(256), size=examples + 1, replace=False).tolist()
        pairs = [(int(value), _bit_tree_value(tree, int(value))) for value in inputs[:-1]]
        query_value = int(inputs[-1])
        ident = identify_bit_expression(pairs, query_value, max_depth=2)
        answer = f"{_bit_tree_value(tree, query_value):08b}"
        if ident["answer_unique"] and answer in ident["query_outputs"]:
            prompt = (
                "Below are input-output examples for a hidden deterministic 8-bit transformation.\n"
                + "\n".join(f"Input: 0b{value:08b} -> Output: 0b{output:08b}" for value, output in pairs)
                + f"\nNow, determine the output for input: 0b{query_value:08b}"
            )
            return {
                "task_type": "bit_manipulation",
                "source": "deterministic_prior_generator",
                "generator_version": GENERATOR_VERSION,
                "hidden_rule": {"expression": render_bit_tree(tree), "depth": 2, "width": 8, "operations": sorted(set(re.findall(r"\b(?:SHL|SHR|ROTL|ROTR|AND|OR|XOR|NOT)\b", render_bit_tree(tree))))},
                "prompt": prompt,
                "answer": answer,
                "trace": {"expression": render_bit_tree(tree), "example_inputs": [v for v, _ in pairs], "example_outputs": [f"{o:08b}" for _, o in pairs], "query_input": query_value, "identifiability": ident},
                "difficulty": "depth_2",
                "deduce_or_guess": "deduce",
            }
    raise RuntimeError("bit_generator_failed_to_find_identifiable_sample")


def verify_bit_sample(sample: dict[str, object]) -> dict[str, object]:
    prompt = str(sample.get("prompt", ""))
    pairs = [(int(value, 2), int(output, 2)) for value, output in re.findall(r"Input: 0b([01]{8}) -> Output: 0b([01]{8})", prompt)]
    query_match = re.search(r"output for input: 0b([01]{8})", prompt)
    trace = sample.get("trace", {})
    answer = str(sample.get("answer", "")).removeprefix("0b")
    answer_ok = bool(query_match and re.fullmatch(r"[01]{8}", answer))
    trace_ok = isinstance(trace, dict) and isinstance(trace.get("expression"), str)
    if answer_ok:
        query_value = int(query_match.group(1), 2)
        try:
            expression = str(trace["expression"])
            answer_ok = all(_evaluate_bit_expression(expression, value) == output for value, output in pairs)
            answer_ok = answer_ok and _evaluate_bit_expression(expression, query_value) == int(answer, 2)
            trace_ok = trace_ok and trace.get("example_inputs") == [v for v, _ in pairs] and trace.get("query_input") == query_value
        except (ValueError, TypeError, IndexError):
            answer_ok = False
            trace_ok = False
    ident = identify_bit_expression(pairs, int(query_match.group(1), 2), max_depth=2) if query_match else {"query_output_count": 0, "answer_unique": False, "expression_unique": False}
    return {"answer_ok": bool(answer_ok), "trace_ok": bool(trace_ok), "identifiability": ident, "difficulty_ok": str(sample.get("difficulty", "")).startswith("depth_"), "solver_status": "independent_parser_and_finite_enumeration"}


In [ ]:
# Equation Numeric: 24-rule prior, independent solver, and deterministic verifier.
NUMERIC_EXPRESSION_RE = re.compile(r"^\d{2}[^\d]\d{2}$")
NUMERIC_OPERATOR_POOL = tuple("+-*/|\\@$%&!?^#~`")


def _numeric_operands(expression: str, parse_name: str) -> tuple[str, str]:
    if NUMERIC_EXPRESSION_RE.fullmatch(expression) is None:
        raise ValueError("invalid_numeric_expression")
    left_text, right_text = expression[:2], expression[3:]
    if parse_name == "AB_CD":
        return left_text, right_text
    if parse_name == "BA_DC":
        return left_text[::-1], right_text[::-1]
    raise ValueError("unsupported_parse")


def _numeric_operation(operation: str, left_text: str, right_text: str) -> int | str | None:
    left, right = int(left_text), int(right_text)
    if operation == "concat_lr": return left_text + right_text
    if operation == "concat_rl": return right_text + left_text
    if operation == "add": return left + right
    if operation == "add_p1": return left + right + 1
    if operation == "add_m1": return left + right - 1
    if operation == "sub_lr": return left - right
    if operation == "sub_rl": return right - left
    if operation == "abs_sub": return abs(left - right)
    if operation == "mul": return left * right
    if operation == "mul_p1": return left * right + 1
    if operation == "mul_m1": return left * right - 1
    if operation == "max_mod_min":
        smaller = min(left, right)
        return None if smaller == 0 else max(left, right) % smaller
    raise ValueError(f"unsupported_operation:{operation}")


def _numeric_sign_token(operator: str) -> str:
    return "-" if operator == "-" else operator


def _numeric_format(value: int | str, format_name: str, operator: str) -> str | None:
    if format_name == "raw":
        if isinstance(value, str): return value
        return str(value) if value >= 0 else f"{_numeric_sign_token(operator)}{abs(value)}"
    if not isinstance(value, int): return None
    if format_name == "rev": return str(value)[::-1] if value >= 0 else None
    if format_name == "sign_prefix_rev": return str(value)[::-1] if value >= 0 else f"{_numeric_sign_token(operator)}{str(abs(value))[::-1]}"
    if format_name == "zpad2": return f"{_numeric_sign_token(operator)}{abs(value):02d}" if value < 0 else f"{value:02d}"
    if format_name == "neg_prefix_rev": return "0" if value == 0 else (None if value > 0 else f"{_numeric_sign_token(operator)}{str(abs(value))[::-1]}")
    if format_name == "always_prefix_rev": return "0" if value == 0 else f"{_numeric_sign_token(operator)}{str(abs(value))[::-1]}"
    if format_name == "sign_prefix_raw": return "0" if value == 0 else f"{operator}{abs(value)}"
    if format_name == "sign_prefix_zpad2": return "00" if value == 0 else f"{operator}{abs(value):02d}"
    if format_name == "neg_suffix_rev": return "0" if value == 0 else f"{str(abs(value))[::-1]}{operator}"
    if format_name == "always_suffix_rev": return "0" if value == 0 else f"{str(abs(value))[::-1]}{operator}"
    if format_name == "rev_or_neg_suffix": return str(value)[::-1] if value >= 0 else f"{str(abs(value))[::-1]}{operator}"
    raise ValueError(f"unsupported_format:{format_name}")


def numeric_rule_output(rule_name: str, expression: str) -> str | None:
    spec = NUMERIC24_RULE_BY_NAME[rule_name]
    left_text, right_text = _numeric_operands(expression, spec["parse"])
    operation_value = _numeric_operation(spec["operation"], left_text, right_text)
    return None if operation_value is None else _numeric_format(operation_value, spec["format"], expression[2])


def _parse_numeric_prompt(prompt: str) -> tuple[list[tuple[str, str]], str] | None:
    examples, query = [], None
    for line in str(prompt).splitlines():
        line = line.strip()
        if line.startswith("Now, determine the result for:"):
            query = line.split(":", 1)[1].strip()
        elif " = " in line:
            expression, output = line.split(" = ", 1)
            if NUMERIC_EXPRESSION_RE.fullmatch(expression.strip()): examples.append((expression.strip(), output.strip()))
    return (examples, query) if examples and query else None


def _numeric_candidate_rules(examples: list[tuple[str, str]], operator: str) -> list[str]:
    same_operator = [(expression, output) for expression, output in examples if expression[2] == operator]
    candidates = []
    for rule in NUMERIC24_RULES:
        if all(numeric_rule_output(rule["name"], expression) == output for expression, output in same_operator):
            candidates.append(rule["name"])
    return candidates


def _numeric_valid_assignments(candidates_by_operator: dict[str, list[str]], operators: list[str]) -> list[dict[str, str]]:
    assignments = []
    for mode in ("normal", "flip"):
        selected = {}
        used_groups = set()
        def backtrack(index: int) -> None:
            if index == len(operators):
                assignments.append({"mode": mode, "rules": dict(selected)})
                return
            operator = operators[index]
            for rule_name in candidates_by_operator[operator]:
                spec = NUMERIC24_RULE_BY_NAME[rule_name]
                if spec["family"] != "join" and spec["family"] != mode:
                    continue
                group = spec["group"]
                if group != "join_like" and group in used_groups:
                    continue
                selected[operator] = rule_name
                if group != "join_like": used_groups.add(group)
                backtrack(index + 1)
                if group != "join_like": used_groups.remove(group)
                selected.pop(operator, None)
        backtrack(0)
    return assignments


def solve_equation_numeric_prompt(prompt: str) -> dict[str, object]:
    parsed = _parse_numeric_prompt(prompt)
    if parsed is None: return {"status": "invalid_prompt", "query_outputs": []}
    examples, query_expression = parsed
    query_operator = query_expression[2]
    example_operators = []
    for expression, _ in examples:
        if expression[2] not in example_operators: example_operators.append(expression[2])
    operators = example_operators + ([] if query_operator in example_operators else [query_operator])
    candidates_by_operator = {operator: _numeric_candidate_rules(examples, operator) for operator in operators}
    if any(not values for values in candidates_by_operator.values()):
        return {"status": "no_compatible_rule", "query_outputs": [], "candidates_by_operator": candidates_by_operator}
    assignments = _numeric_valid_assignments(candidates_by_operator, operators)
    query_outputs = sorted({numeric_rule_output(assignment["rules"][query_operator], query_expression) for assignment in assignments})
    query_outputs = [value for value in query_outputs if value is not None]
    selected = assignments[0] if assignments else None
    return {
        "status": "solved" if assignments else "no_valid_global_assignment",
        "query_operator": query_operator,
        "query_operator_in_examples": query_operator in example_operators,
        "candidates_by_operator": candidates_by_operator,
        "valid_assignment_count": len(assignments),
        "valid_assignments": assignments[:20],
        "query_outputs": query_outputs,
        "answer_unique": len(query_outputs) == 1,
        "selected_assignment": selected,
    }


def _numeric_truth_output(rule_name: str, expression: str) -> str | None:
    """Generator-side arithmetic implementation kept separate from the solver API."""
    spec = NUMERIC24_RULE_BY_NAME[rule_name]
    left, right = expression[:2], expression[3:]
    if spec["parse"] == "BA_DC": left, right = left[::-1], right[::-1]
    a, b = int(left), int(right)
    operation = spec["operation"]
    if operation == "concat_lr": value = left + right
    elif operation == "concat_rl": value = right + left
    elif operation == "add": value = a + b
    elif operation == "add_p1": value = a + b + 1
    elif operation == "add_m1": value = a + b - 1
    elif operation == "sub_lr": value = a - b
    elif operation == "sub_rl": value = b - a
    elif operation == "abs_sub": value = abs(a - b)
    elif operation == "mul": value = a * b
    elif operation == "mul_p1": value = a * b + 1
    elif operation == "mul_m1": value = a * b - 1
    elif operation == "max_mod_min":
        value = None if min(a, b) == 0 else max(a, b) % min(a, b)
    else: raise ValueError(operation)
    return None if value is None else _numeric_format(value, spec["format"], expression[2])


def _numeric_random_expression(rng: np.random.Generator, operator: str, avoid_zero_divisor: bool = False) -> str:
    left, right = int(rng.integers(0, 100)), int(rng.integers(0, 100))
    if avoid_zero_divisor and right == 0: right = int(rng.integers(1, 100))
    return f"{left:02d}{operator}{right:02d}"


def _numeric_prompt_text(example_pairs: list[tuple[str, str]], query_expression: str) -> str:
    return (
        "In this numeric equation task, each operator has a hidden transformation rule.\n"
        "The same rule assignment is used consistently within this prompt.\n"
        + "\n".join(f"{expression} = {output}" for expression, output in example_pairs)
        + f"\nNow, determine the result for: {query_expression}"
    )


def _numeric_hidden_assignment(rng: np.random.Generator, mode: str) -> dict[str, str]:
    pools = {}
    for group in ("sub_or_mod_like", "add_like", "mul_like"):
        choices = [rule["name"] for rule in NUMERIC24_RULES if rule["family"] == mode and rule["group"] == group]
        pools[group] = str(rng.choice(choices))
    pools["join_like"] = str(rng.choice(["join", "join_swap"]))
    return pools


def generate_equation_numeric_sample(rng: np.random.Generator, guess: bool = False) -> dict[str, object]:
    for _ in range(500):
        mode = str(rng.choice(["normal", "flip"]))
        by_group = _numeric_hidden_assignment(rng, mode)
        if guess:
            example_operators = list(rng.choice(np.array(NUMERIC_OPERATOR_POOL), size=3, replace=False))
            query_operator = next(operator for operator in NUMERIC_OPERATOR_POOL if operator not in example_operators)
            hidden = {operator: by_group[group] for operator, group in zip(example_operators, ("sub_or_mod_like", "add_like", "mul_like"))}
            hidden[query_operator] = by_group["join_like"]
        else:
            operators = list(rng.choice(np.array(NUMERIC_OPERATOR_POOL), size=4, replace=False))
            hidden = {operator: by_group[group] for operator, group in zip(operators, ("sub_or_mod_like", "add_like", "mul_like", "join_like"))}
            query_operator = str(rng.choice(np.array(operators)))
            example_operators = operators
        pairs = []
        for operator in example_operators:
            rule_name = hidden[operator]
            for _example in range(5):
                expression = _numeric_random_expression(rng, operator, avoid_zero_divisor="mod" in NUMERIC24_RULE_BY_NAME[rule_name]["operation"])
                output = _numeric_truth_output(rule_name, expression)
                if output is not None: pairs.append((expression, output))
        if guess:
            query_expression = f"{int(rng.integers(10, 100)):02d}{query_operator}{int(rng.integers(10, 100)):02d}"
            # Equal operands make join and reverse-join semantically equivalent for this query.
            same = int(rng.integers(10, 100))
            query_expression = f"{same:02d}{query_operator}{same:02d}"
        else:
            query_expression = _numeric_random_expression(rng, query_operator, avoid_zero_divisor=False)
        answer = _numeric_truth_output(hidden[query_operator], query_expression)
        if answer is None: continue
        prompt = _numeric_prompt_text(pairs, query_expression)
        resolution = solve_equation_numeric_prompt(prompt)
        if resolution.get("answer_unique") and answer in resolution.get("query_outputs", []):
            return {
                "task_type": "equation_numeric",
                "source": "deterministic_prior_generator",
                "generator_version": GENERATOR_VERSION,
                "hidden_rule": {"mode": mode, "operator_to_rule": hidden, "rule_formulas": {name: NUMERIC24_FORMULAS[name] for name in hidden.values()}},
                "prompt": prompt,
                "answer": answer,
                "trace": {"solver": "24-rule enumeration with family/group constraints", "query_operator": query_operator, "query_operator_in_examples": not guess, "candidate_rules_by_operator": resolution["candidates_by_operator"], "valid_assignment_count": resolution["valid_assignment_count"], "selected_assignment": resolution["selected_assignment"], "query_outputs": resolution["query_outputs"]},
                "difficulty": "guess_global_constraint" if guess else "deduce_multi_operator",
                "deduce_or_guess": "guess" if guess else "deduce",
            }
    raise RuntimeError(f"numeric_generator_failed:{'guess' if guess else 'deduce'}")


def _numeric_verify_output(rule_name: str, expression: str) -> str | None:
    """Verifier-side implementation; it parses and recomputes the rule independently."""
    spec = NUMERIC24_RULE_BY_NAME.get(rule_name)
    if spec is None or NUMERIC_EXPRESSION_RE.fullmatch(expression) is None: return None
    left_text, right_text = expression[:2], expression[3:]
    if spec["parse"] == "BA_DC": left_text, right_text = left_text[::-1], right_text[::-1]
    left, right = int(left_text), int(right_text)
    op = spec["operation"]
    if op == "concat_lr": value = left_text + right_text
    elif op == "concat_rl": value = right_text + left_text
    elif op == "add": value = left + right
    elif op == "add_p1": value = left + right + 1
    elif op == "add_m1": value = left + right - 1
    elif op == "sub_lr": value = left - right
    elif op == "sub_rl": value = right - left
    elif op == "abs_sub": value = abs(left - right)
    elif op == "mul": value = left * right
    elif op == "mul_p1": value = left * right + 1
    elif op == "mul_m1": value = left * right - 1
    elif op == "max_mod_min": value = None if min(left, right) == 0 else max(left, right) % min(left, right)
    else: return None
    return None if value is None else _numeric_format(value, spec["format"], expression[2])


def verify_equation_numeric_sample(sample: dict[str, object]) -> dict[str, object]:
    parsed = _parse_numeric_prompt(str(sample.get("prompt", "")))
    if parsed is None: return {"answer_ok": False, "trace_ok": False, "identifiability": {"answer_unique": False}, "difficulty_ok": False}
    examples, query_expression = parsed
    answer = str(sample.get("answer", ""))
    # The verifier uses only the public prompt and the independently enumerated
    # prior. hidden_rule is retained as audit metadata and is deliberately not
    # consulted to establish correctness.
    resolution = solve_equation_numeric_prompt(str(sample.get("prompt", "")))
    answer_ok = bool(resolution.get("answer_unique", False)) and answer in resolution.get("query_outputs", [])
    ident = {"valid_assignment_count": resolution.get("valid_assignment_count", 0), "query_output_count": len(resolution.get("query_outputs", [])), "query_outputs": resolution.get("query_outputs", []), "answer_unique": bool(resolution.get("answer_unique", False)), "rule_assignment_unique": resolution.get("valid_assignment_count", 0) == 1}
    trace = sample.get("trace", {})
    trace_ok = isinstance(trace, dict) and trace.get("query_outputs") == resolution.get("query_outputs") and trace.get("query_operator") == query_expression[2]
    return {"answer_ok": bool(answer_ok), "trace_ok": bool(trace_ok), "identifiability": ident, "difficulty_ok": str(sample.get("difficulty", "")).startswith(("deduce_", "guess_")), "solver_status": "24_rule_global_assignment"}


In [ ]:
# Unified screening, export, and optional NeMo Data Designer Preview.
import os

SYNTHETIC_SCHEMA_FIELDS = ["task_type", "source", "generator_version", "hidden_rule", "prompt", "answer", "trace", "difficulty", "deduce_or_guess"]


def synthetic_prompt_fingerprint(text: object) -> str:
    # Keep numeric parameters: masking all numbers would reject every sample sharing a template.
    normalized = re.sub(r"\s+", " ", str(text).lower().strip())
    return prompt_hash(normalized)


def screen_synthetic_sample(sample: dict[str, object], seen_exact: set[str], seen_normalized: set[str]) -> dict[str, object]:
    reasons = []
    sample["deduce_or_guess"] = sample.get("deduce_or_guess") or ("deduce" if sample.get("task_type") != "equation_numeric" else "unknown")
    schema_ok = all(sample.get(field) not in (None, "", {}) for field in SYNTHETIC_SCHEMA_FIELDS)
    if not schema_ok: reasons.append("schema_invalid")
    if sample.get("task_type") not in TARGET_TASKS: reasons.append("unsupported_task")
    if sample.get("source") != "deterministic_prior_generator": reasons.append("unexpected_source")
    exact_key = prompt_hash(sample.get("prompt", ""))
    normalized_key = synthetic_prompt_fingerprint(sample.get("prompt", ""))
    if exact_key in seen_exact: reasons.append("duplicate_prompt")
    if normalized_key in seen_normalized: reasons.append("near_duplicate_prompt")
    if sample.get("task_type") == "gravity": verification = verify_gravity_sample(sample)
    elif sample.get("task_type") == "bit_manipulation": verification = verify_bit_sample(sample)
    elif sample.get("task_type") == "equation_numeric": verification = verify_equation_numeric_sample(sample)
    else: verification = {"answer_ok": False, "trace_ok": False, "identifiability": {"answer_unique": False}, "difficulty_ok": False}
    if not verification.get("answer_ok", False): reasons.append("answer_verification_failed")
    if not verification.get("trace_ok", False): reasons.append("trace_verification_failed")
    ident = verification.get("identifiability", {})
    if not ident.get("answer_unique", False): reasons.append("answer_not_identifiable")
    if not verification.get("difficulty_ok", False): reasons.append("difficulty_audit_failed")
    accepted = not reasons
    record = {
        "candidate_index": sample.get("candidate_index", -1),
        "task_type": sample.get("task_type", ""),
        "source": sample.get("source", ""),
        "generator_version": sample.get("generator_version", ""),
        "deduce_or_guess": sample.get("deduce_or_guess", "deduce" if sample.get("task_type") != "equation_numeric" else "unknown"),
        "prompt_hash": exact_key,
        "normalized_prompt_hash": normalized_key,
        "schema_ok": schema_ok,
        "answer_ok": bool(verification.get("answer_ok", False)),
        "trace_ok": bool(verification.get("trace_ok", False)),
        "identifiability": json.dumps(ident, ensure_ascii=False, sort_keys=True),
        "difficulty": sample.get("difficulty", ""),
        "accepted": accepted,
        "verification_status": "accepted" if accepted else "rejected",
        "rejection_reasons": ";".join(dict.fromkeys(reasons)),
    }
    return record


def _write_jsonl(path: Path, records: list[dict[str, object]]) -> None:
    payload = "\n".join(json.dumps(record, ensure_ascii=False) for record in records)
    path.write_text(payload + ("\n" if payload else ""), encoding="utf-8")


def _data_designer_preview() -> dict[str, object]:
    """Optional, API-checked integration; outputs are never admitted without our verifier."""
    if not RUN_DATA_DESIGNER_PREVIEW:
        return {"status": "skipped_by_default", "reason": "RUN_DATA_DESIGNER_PREVIEW=False"}
    if not os.environ.get("NVIDIA_API_KEY"):
        return {"status": "blocked_missing_NVIDIA_API_KEY", "reason": "设置环境变量 NVIDIA_API_KEY 后再调用 NVIDIA hosted endpoint"}
    try:
        import data_designer.config as dd
        from data_designer.interface import DataDesigner
    except ImportError as exc:
        return {"status": "blocked_missing_data_designer", "reason": repr(exc), "install": "遵循 Data Designer 官方文档安装 data-designer；本 Notebook 不临时安装依赖"}
    try:
        accepted_seed = []
        accepted_path = SYNTHETIC_DIR / "data_designer_seed.jsonl"
        for line in (SYNTHETIC_DIR / "synthetic_accepted.jsonl").read_text(encoding="utf-8").splitlines()[:10]:
            item = json.loads(line)
            # Only linguistic seed fields cross the optional boundary; no answer or hidden rule is sent.
            accepted_seed.append({"task_type": item["task_type"], "prompt": item["prompt"]})
        _write_jsonl(accepted_path, accepted_seed)
        model_configs = [dd.ModelConfig(
            alias="nvidia-text",
            model="nvidia/nemotron-3-nano-30b-a3b",
            provider="nvidia",
            inference_parameters=dd.ChatCompletionInferenceParams(temperature=0.7, top_p=0.95, max_tokens=512),
        )]
        designer = DataDesigner()
        builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)
        builder.with_seed_dataset(dd.LocalFileSeedSource(path=str(accepted_path)), sampling_strategy=dd.SamplingStrategy.SHUFFLE)
        builder.add_column(dd.LLMTextColumnConfig(
            name="prompt_variant",
            model_alias="nvidia-text",
            prompt="Rewrite the following task prompt for clearer natural language while preserving every number, operator, input, and output example exactly. Return only the rewritten prompt.\n{{ prompt }}",
        ))
        designer.validate(builder)
        preview = designer.preview(builder, num_records=min(8, len(accepted_seed)))
        dataset = preview.dataset
        if hasattr(dataset, "to_dict"): dataset = dataset.to_dict(orient="records")
        preview_records = dataset if isinstance(dataset, list) else []
        _write_jsonl(SYNTHETIC_DIR / "data_designer_preview.jsonl", preview_records)
        return {"status": "completed", "records": len(preview_records), "verified": False, "reason": "Preview must pass deterministic task verifiers before any merge"}
    except Exception as exc:
        return {"status": "blocked_runtime_error", "reason": repr(exc), "verified": False}


def run_synthetic_pilot() -> dict[str, object]:
    rng = np.random.default_rng(SEED)
    generated = []
    failures = []
    task_order = ["gravity", "equation_numeric", "bit_manipulation"]
    for task_type in task_order:
        for index in range(PILOT_CANDIDATES_PER_TASK):
            candidate_index = f"{task_type}-{index:04d}"
            try:
                if task_type == "gravity": sample = generate_gravity_sample(rng)
                elif task_type == "bit_manipulation": sample = generate_bit_sample(rng)
                else: sample = generate_equation_numeric_sample(rng, guess=(index % 5 == 0))
                sample["candidate_index"] = candidate_index
                generated.append(sample)
            except Exception as exc:
                failures.append({"candidate_index": candidate_index, "task_type": task_type, "error": repr(exc)})
                generated.append({
                    "candidate_index": candidate_index, "task_type": task_type, "source": "deterministic_prior_generator", "generator_version": GENERATOR_VERSION,
                    "hidden_rule": {}, "prompt": "", "answer": "", "trace": {}, "difficulty": "generation_failed", "deduce_or_guess": "unknown",
                    "generation_error": repr(exc),
                })
    seen_exact, seen_normalized = set(), set()
    screening = []
    accepted, rejected = [], []
    for sample in generated:
        record = screen_synthetic_sample(sample, seen_exact, seen_normalized)
        screening.append(record)
        exact_key, normalized_key = record["prompt_hash"], record["normalized_prompt_hash"]
        seen_exact.add(exact_key); seen_normalized.add(normalized_key)
        sample["verification_status"] = record["verification_status"]
        sample["screening"] = record
        (accepted if record["accepted"] else rejected).append(sample)
    _write_jsonl(SYNTHETIC_DIR / "synthetic_candidates.jsonl", generated)
    _write_jsonl(SYNTHETIC_DIR / "synthetic_accepted.jsonl", accepted)
    _write_jsonl(SYNTHETIC_DIR / "synthetic_rejected.jsonl", rejected)
    screening_df = pd.DataFrame(screening)
    screening_df.to_csv(SYNTHETIC_DIR / "synthetic_screening.csv", index=False)
    screening_df.to_csv(OUTPUT_DIR / "synthetic_pilot_screening.csv", index=False)
    by_task = {}
    for task_type, group in screening_df.groupby("task_type", sort=True):
        reasons = Counter(reason for text in group["rejection_reasons"] for reason in str(text).split(";") if reason)
        by_task[task_type] = {"candidates": int(len(group)), "accepted": int(group["accepted"].sum()), "rejected": int((~group["accepted"]).sum()), "acceptance_rate": float(group["accepted"].mean()), "rejection_reasons": dict(reasons)}
    hidden_rule_counts = Counter()
    bit_operation_counts = Counter()
    gravity_values = []
    for sample in generated:
        hidden = sample.get("hidden_rule", {})
        if sample.get("task_type") == "equation_numeric": hidden_rule_counts.update(hidden.get("operator_to_rule", {}).values())
        elif sample.get("task_type") == "bit_manipulation": bit_operation_counts.update(hidden.get("operations", []))
        elif sample.get("task_type") == "gravity" and hidden.get("g") is not None: gravity_values.append(hidden["g"])
    data_designer_status = _data_designer_preview()
    metadata = {
        "schema_version": "nemotron-course-synthetic/v2", "seed": SEED, "generated_at_utc": datetime.now(timezone.utc).isoformat(),
        "requested_candidates_per_task": PILOT_CANDIDATES_PER_TASK, "task_order": task_order, "generator_version": GENERATOR_VERSION,
        "train_prior_only": True, "validation_or_holdout_used": False, "data_designer": data_designer_status,
        "data_designer_official_api": {"package": "data-designer", "imports": ["data_designer.config", "data_designer.interface.DataDesigner"], "seed_api": "LocalFileSeedSource + DataDesignerConfigBuilder.with_seed_dataset", "preview_api": "DataDesigner.preview(builder, num_records=...)"},
        "generation_failures": failures,
    }
    (SYNTHETIC_DIR / "generation_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
    deduce_guess_by_task = {
        task_type: {k: int(v) for k, v in group["deduce_or_guess"].value_counts(dropna=False).to_dict().items()}
        for task_type, group in screening_df.groupby("task_type", sort=True)
    }
    statistics = {
        "mode": "pilot", "seed": SEED, "requested_candidates_per_task": PILOT_CANDIDATES_PER_TASK, "generated": len(generated), "accepted": len(accepted), "rejected": len(rejected),
        "by_task": by_task, "deduce_guess_counts": {k: int(v) for k, v in screening_df["deduce_or_guess"].value_counts(dropna=False).to_dict().items()}, "deduce_guess_by_task": deduce_guess_by_task,
        "rule_coverage": {"equation_numeric": dict(hidden_rule_counts), "bit_operations": dict(bit_operation_counts), "gravity_g_unique": sorted(set(gravity_values))},
        "rejection_reasons": dict(Counter(reason for text in screening_df["rejection_reasons"] for reason in str(text).split(";") if reason)),
        "duplicate_audit": {"exact_duplicate_rows": int(len(screening_df) - screening_df["prompt_hash"].nunique()), "normalized_duplicate_rows": int(len(screening_df) - screening_df["normalized_prompt_hash"].nunique())},
        "data_designer": data_designer_status,
        "artifacts": {name: f"data/processed/synthetic/{name}" for name in ["synthetic_candidates.jsonl", "synthetic_accepted.jsonl", "synthetic_rejected.jsonl", "synthetic_screening.csv", "generation_metadata.json"]},
    }
    (SYNTHETIC_DIR / "synthetic_statistics.json").write_text(json.dumps(statistics, ensure_ascii=False, indent=2), encoding="utf-8")
    (OUTPUT_DIR / "synthetic_statistics.json").write_text(json.dumps(statistics, ensure_ascii=False, indent=2), encoding="utf-8")
    return statistics


if RUN_SYNTHETIC:
    synthetic_pilot_summary = run_synthetic_pilot()
else:
    synthetic_pilot_summary = {"mode": "disabled", "reason": "RUN_SYNTHETIC=False"}
print(json.dumps(synthetic_pilot_summary, ensure_ascii=False, indent=2))


## 8. 交付检查与未完成事项

本节汇总可复核文件和真实 Pilot 结果。Baseline 的准确率只有在 `RUN_BASELINE=True` 且 vLLM 实际完成后才会出现；当前仍默认关闭。5090 上已完成单样本 vLLM smoke 的配置验证时，应以 `data/processed/vllm_smoke_result.json` 的真实状态为准。Data Designer Preview 是独立可选分支，不通过确定性 Verifier 的结果不会进入合成数据集。


In [ ]:
expected_artifacts = [
    OUTPUT_DIR / "real_cleaned.csv",
    OUTPUT_DIR / "classification_audit.csv",
    OUTPUT_DIR / "dataset_statistics.json",
    OUTPUT_DIR / "split_manifest.json",
    SPLIT_DIR / "train.csv",
    SPLIT_DIR / "validation.csv",
    SPLIT_DIR / "holdout_test.csv",
    OUTPUT_DIR / "synthetic_pilot_screening.csv",
    OUTPUT_DIR / "synthetic_statistics.json",
    SYNTHETIC_DIR / "synthetic_candidates.jsonl",
    SYNTHETIC_DIR / "synthetic_accepted.jsonl",
    SYNTHETIC_DIR / "synthetic_rejected.jsonl",
    SYNTHETIC_DIR / "synthetic_screening.csv",
    SYNTHETIC_DIR / "generation_metadata.json",
]
missing = [str(path) for path in expected_artifacts if not path.exists()]
if missing:
    raise AssertionError(f"缺少产物：{missing}")
synthetic_screening_df = pd.read_csv(SYNTHETIC_DIR / "synthetic_screening.csv")
assert len(synthetic_screening_df) == 3 * PILOT_CANDIDATES_PER_TASK
assert synthetic_screening_df["verification_status"].notna().all()
assert set(synthetic_screening_df["task_type"]) == TARGET_TASKS
print(json.dumps({
    "artifact_count": len(expected_artifacts),
    "missing": missing,
    "cleaned_rows": int(len(cleaned_df)),
    "baseline_status": baseline_status["status"],
    "synthetic_candidates": int(len(synthetic_screening_df)),
    "synthetic_accepted": int(synthetic_screening_df["accepted"].sum()),
    "synthetic_rejected": int((~synthetic_screening_df["accepted"]).sum()),
    "synthetic_by_task": synthetic_screening_df.groupby("task_type")["accepted"].agg(["count", "sum"]).to_dict(orient="index"),
    "data_designer_status": synthetic_pilot_summary.get("data_designer"),
    "next_steps": [
        "Baseline 仍保持 RUN_BASELINE=False；在 RTX 5090 空闲时单独执行 vLLM smoke。",
        "Data Designer 只负责语言表达变体；任何 preview 结果必须重新通过确定性 Verifier。",
        "当前 bit DSL 是课程用 8-bit 有限子集，不宣称覆盖比赛全部位操作规则。",
    ],
}, ensure_ascii=False, indent=2))
